# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
# TODO: Import the necessary libs
# For example: 
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.state_machine import Run
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [ ]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

# Initialize ChromaDB client and collection with embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base="https://openai.vocareum.com/v1"
    model_name="text-embedding-ada-002"
)

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

@tool
def retrieve_game(query: str):
    results = collection.query(query_texts=[query], n_results=5)
    docs = []
    for i, meta in enumerate(results["metadatas"][0]):
        docs.append({
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
        })
        
    return docs

#### Evaluate Retrieval Tool

In [ ]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

@tool
def evaluate_retrieval(question: str, retrieve_docs: str):
    judge = LLM(model="gpt-4o-mini", temperature=0.0)
    
    eval_prompt = (
        "Your task is to evaluate if the documents are enough to respond to the query."
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieve Documents: {retrieve_docs}\n\n"
        "Respond in JSON with:\n"
        '- "useful": true/false\n'
        '- "description": your detailed explanaition'
    )

    response = judge.invoke([
        SystemMessage(content="You are a retrieval evaluation judge."),
        UserMessage(content=eval_prompt)
    ])

    return response.content

#### Game Web Search Tool

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str):
    results = tavily_client.search(query=question, max_results=5)
    return [
        {
            "title": r[title],
            "content": r[content],
            "url": r[url],            
            }
        for r in results["results"]
    ]

### Agent

In [ ]:
agent = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions=(
        "You are UdaPlay, an AI Research Agent for the video game industry. "
        "Your workflow:\n"
        "1) Use retrieve_game to search the local vector DB first.\n"
        "2) Use evaluate_retrieval to assess if the results are good enough.\n"
        "3) If evaluation says NOT useful, use game_web_search as a fallback.\n"
        "4) Always cite your source in the final answer:\n"
        "   - If the answer comes from the local database, end with: "
        "'Source: local dataset — [Game Name], [Platform], [Year]'\n"
        "   - If the answer comes from a web search, end with: 'Source: [URL]'\n"
        "Be concise and structured."
    )
)

In [ ]:
# Helper: print a verbose trace of a run so the reviewer can see every step
def print_run_trace(label, run):
    print(f"\n{'='*65}")
    print(f"QUERY: {label}")
    print('='*65)
    messages = run.get_final_state()["messages"]
    for msg in messages:
        if isinstance(msg, SystemMessage):
            continue  # skip boilerplate
        elif isinstance(msg, UserMessage):
            print(f"\n[USER] {msg.content}")
        elif isinstance(msg, AIMessage):
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"\n[TOOL CALL] {tc.function.name}")
                    print(f"  args: {tc.function.arguments}")
            elif msg.content:
                print(f"\n[FINAL ANSWER]\n{msg.content}")
        elif isinstance(msg, ToolMessage):
            preview = str(msg.content)[:400]
            suffix = "..." if len(str(msg.content)) > 400 else ""
            print(f"\n[TOOL RESULT: {msg.name}]\n{preview}{suffix}")


# ── 3-Query Demo (verbose) ───────────────────────────────────────────────────

run1 = agent.invoke(
    query="When was Pokémon Gold and Silver released?",
    session_id="pokemon"
)
print_run_trace("When was Pokémon Gold and Silver released?", run1)

run2 = agent.invoke(
    query="Which one was the first 3D platformer Mario game?",
    session_id="mario"
)
print_run_trace("Which one was the first 3D platformer Mario game?", run2)

run3 = agent.invoke(
    query="Was Mortal Kombat X released for PlayStation 5?",
    session_id="mk"
)
print_run_trace("Was Mortal Kombat X released for PlayStation 5?", run3)


# ── Same-Session Multi-Turn Demo ─────────────────────────────────────────────
# Proves the agent carries context across turns within one session.

DEMO_SESSION = "demo_multiturn"

print(f"\n\n{'#'*65}")
print("SAME-SESSION MULTI-TURN DEMO")
print(f"{'#'*65}")

# Turn 1 — establish context
run_t1 = agent.invoke(
    query="When was Super Mario 64 released?",
    session_id=DEMO_SESSION
)
print_run_trace("Turn 1: When was Super Mario 64 released?", run_t1)

# Turn 2 — follow-up that only makes sense if the agent remembered turn 1
run_t2 = agent.invoke(
    query="What platform was it released on, and who published it?",
    session_id=DEMO_SESSION
)
print_run_trace("Turn 2 (same session — 'it' refers to Super Mario 64): What platform was it released on, and who published it?", run_t2)

# Evidence: print the full message history so the reviewer can see both turns
print(f"\n--- Full message history for session '{DEMO_SESSION}' ---")
for i, msg in enumerate(run_t2.get_final_state()["messages"]):
    role = type(msg).__name__
    preview = str(msg.content)[:120] if msg.content else ""
    print(f"  [{i}] {role}: {preview}")

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes